# PostgreSQL → Databricks Incremental Migration
## Restartable Bronze ingestion with watermark control, deterministic batch IDs, idempotency, and recovery

**Purpose**

```text
Local PostgreSQL
      ↓ JDBC
Databricks control state
      ↓
Bounded incremental extraction
      ↓
Bronze Delta — append-only history
      ↓
Control state commit
      ↓
Ingestion audit log
```

This notebook implements **watermark-based incremental ingestion**, not true source CDC.

- **Full load:** initial snapshot.
- **Incremental load:** rows inside the next `updated_at` window.
- **Bronze:** append-only ingestion history.
- **Control state:** latest successfully committed watermark.
- **Ingestion log:** audit history of batch/run attempts.
- **Idempotency:** retrying the same logical batch does not append it twice.


## Design decision: bounded extraction windows

A weak implementation uses only:

```sql
WHERE updated_at > last_watermark
```

This notebook first captures an **upper watermark** from the source:

```text
start_watermark = last successful watermark
end_watermark   = MAX(updated_at) at extraction start
```

Then it extracts:

```sql
updated_at > start_watermark
AND updated_at <= end_watermark
```

This gives a deterministic processing window. Records changed after `end_watermark` wait for the next batch.


## 01 — Configuration

After a compute/session restart, rerun this cell. In production, use Databricks secret/credential management instead of storing passwords in notebook source.

In [0]:
from datetime import datetime
import hashlib
import uuid

from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Source connection
# ---------------------------------------------------------------------------

driver = "org.postgresql.Driver"

# Example:
# jdbc_url = "jdbc:postgresql://0.tcp.in.ngrok.io:26324/demo"
jdbc_url = "jdbc:postgresql://0.tcp.in.ngrok.io:29031/demo"

username = "postgres"
password = "root"

# ---------------------------------------------------------------------------
# Source and target identifiers
# ---------------------------------------------------------------------------

SOURCE_SYSTEM = "PostgreSQL"
SOURCE_SCHEMA = "public"
SOURCE_TABLE = "customers"

BRONZE_TABLE = "migration.bronze.customers"
CONTROL_TABLE = "migration.control.ingestion_state"
LOG_TABLE = "migration.control.ingestion_log"

print("Configuration loaded.")
print("Source :", f"{SOURCE_SCHEMA}.{SOURCE_TABLE}")
print("Bronze :", BRONZE_TABLE)


## 02 — Create persistent control tables

These Delta tables survive compute/session termination.

### `ingestion_state`
Stores **only the latest successful state** for each source table.

### `ingestion_log`
Stores **every batch/run attempt**, including failures and retries.


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS migration.control")
spark.sql("CREATE SCHEMA IF NOT EXISTS migration.bronze")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_processed_at TIMESTAMP,
    last_row_count BIGINT
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    source_system STRING,
    source_table STRING,
    batch_id STRING,
    run_id STRING,
    start_watermark TIMESTAMP,
    end_watermark TIMESTAMP,
    status STRING,
    row_count BIGINT,
    batch_started_at TIMESTAMP,
    batch_completed_at TIMESTAMP
)
USING DELTA
""")

print("Control tables are ready.")


## 03 — Read the latest successful control state

The control table is the authoritative source of the next starting watermark. We do not calculate the permanent watermark from Bronze.

In [0]:
state_df = spark.sql(f"""
SELECT
    last_watermark,
    last_batch_id,
    last_run_id,
    last_status,
    last_processed_at,
    last_row_count
FROM {CONTROL_TABLE}
WHERE source_system = '{SOURCE_SYSTEM}'
  AND source_table = '{SOURCE_TABLE}'
  AND last_status = 'SUCCESS'
""")

state_rows = state_df.collect()

if len(state_rows) > 1:
    raise RuntimeError(
        f"Expected at most one successful control row for "
        f"{SOURCE_SYSTEM}.{SOURCE_TABLE}, found {len(state_rows)}."
    )

if len(state_rows) == 0:
    is_first_run = True
    start_watermark = None
    print("Run type: FULL LOAD")
else:
    is_first_run = False
    start_watermark = state_rows[0]["last_watermark"]
    print("Run type: INCREMENTAL")
    print("Start watermark:", start_watermark)


## 04 — Capture a source-side upper watermark

We first ask PostgreSQL for the maximum `updated_at` currently visible.

This gives us:

```text
start_watermark → previous successful position
end_watermark   → upper boundary of this batch
```

The upper boundary prevents a long-running extraction from continuously expanding its own processing window.


In [0]:
watermark_query = f"""
SELECT MAX(updated_at) AS max_updated_at
FROM {SOURCE_SCHEMA}.{SOURCE_TABLE}
"""

source_max_df = (
    spark.read
    .format("jdbc")
    .option("driver", driver)
    .option("url", jdbc_url)
    .option("query", watermark_query)
    .option("user", username)
    .option("password", password)
    .load()
)

end_watermark = source_max_df.collect()[0]["max_updated_at"]

print("Start watermark:", start_watermark)
print("End watermark  :", end_watermark)


## 05 — Build the bounded extraction query

First run reads the initial snapshot up to `end_watermark`. Later runs read only rows strictly after the previous watermark and up to the current upper boundary.

In [0]:
if end_watermark is None:
    source_df = spark.createDataFrame(
        [],
        "customer_id BIGINT"
    )
    query = None
    print("Source has no rows with updated_at.")
else:
    if is_first_run:
        query = f"""
        SELECT *
        FROM {SOURCE_SCHEMA}.{SOURCE_TABLE}
        WHERE updated_at <= '{end_watermark}'
        """
    else:
        query = f"""
        SELECT *
        FROM {SOURCE_SCHEMA}.{SOURCE_TABLE}
        WHERE updated_at > '{start_watermark}'
          AND updated_at <= '{end_watermark}'
        """

    print("Extraction query:")
    print(query)

    source_df = (
        spark.read
        .format("jdbc")
        .option("driver", driver)
        .option("url", jdbc_url)
        .option("query", query)
        .option("user", username)
        .option("password", password)
        .load()
    )

display(source_df)


## 06 — Handle an empty batch

An empty incremental batch is a successful no-op. We do not create a fake batch, append Bronze, or move the watermark.

In [0]:
extracted_count = source_df.count()
print("Rows extracted:", extracted_count)

if extracted_count == 0:
    print("NO-OP: no new/changed records to process.")
else:
    print("A processable batch was found.")


## 07 — Create a deterministic logical batch ID

Do **not** use the current clock as the logical batch identity.

A retry of the same source window must reconstruct the same batch ID.

The logical identity is derived from:

```text
source system + source table + start watermark + end watermark
```

The hash makes the identifier deterministic and compact.


In [0]:
if extracted_count > 0:
    start_token = (
        "INITIAL"
        if start_watermark is None
        else start_watermark.strftime("%Y%m%dT%H%M%S%f")
    )
    end_token = end_watermark.strftime("%Y%m%dT%H%M%S%f")

    logical_key = (
        f"{SOURCE_SYSTEM}|"
        f"{SOURCE_SCHEMA}.{SOURCE_TABLE}|"
        f"{start_watermark}|"
        f"{end_watermark}"
    )

    logical_hash = hashlib.sha256(
        logical_key.encode("utf-8")
    ).hexdigest()[:12]

    batch_id = (
        f"{SOURCE_TABLE.upper()}_"
        f"{start_token}_TO_{end_token}_"
        f"{logical_hash}"
    )

    # A run ID identifies one execution attempt.
    # A retry gets a new run_id but retains the same logical batch_id.
    run_id = str(uuid.uuid4())

    print("Batch ID:", batch_id)
    print("Run ID  :", run_id)


## 08 — Idempotency check

Two checks protect the Bronze write:

1. A successful batch already in the ingestion log means the batch is complete.
2. Rows already present in Bronze mean the Bronze transaction already committed.

This directly handles:

```text
Bronze write succeeds
       ↓
Control commit fails
       ↓
Retry
       ↓
Same batch_id
       ↓
Skip duplicate Bronze append
```


In [0]:
if extracted_count > 0:
    log_match = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {LOG_TABLE}
    WHERE source_system = '{SOURCE_SYSTEM}'
      AND source_table = '{SOURCE_TABLE}'
      AND batch_id = '{batch_id}'
      AND status = 'SUCCESS'
    """).collect()[0]["cnt"]

    bronze_match = (
        spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {BRONZE_TABLE}
        WHERE _batch_id = '{batch_id}'
        """).collect()[0]["cnt"]
        if spark.catalog.tableExists(BRONZE_TABLE)
        else 0
    )

    batch_already_complete = log_match > 0
    bronze_already_written = bronze_match > 0

    print("Successful log match :", log_match)
    print("Bronze row match     :", bronze_match)

    if batch_already_complete:
        print("Batch already completed successfully.")
    elif bronze_already_written:
        print("Bronze already contains this batch. Recovery path.")
    else:
        print("New batch. Bronze write required.")


## 09 — Prepare Bronze records

Bronze is append-only historical ingestion. We preserve every successfully ingested version. Current-state/SCD logic belongs downstream in Silver.

In [0]:
if extracted_count > 0 and not batch_already_complete:
    bronze_df = (
        source_df
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_run_id", F.lit(run_id))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_source_table", F.lit(SOURCE_TABLE))
    )

    display(bronze_df)


## 10 — Write Bronze

First run creates the baseline. Later runs append new logical batches. We never MERGE or overwrite historical Bronze batches during normal incremental processing.

In [0]:
if (
    extracted_count > 0
    and not batch_already_complete
    and not bronze_already_written
):
    write_mode = "overwrite" if is_first_run else "append"

    (
        bronze_df
        .write
        .format("delta")
        .mode(write_mode)
        .saveAsTable(BRONZE_TABLE)
    )

    print(
        f"Bronze write completed: {extracted_count} rows "
        f"(mode={write_mode})"
    )

    bronze_already_written = True


## 11 — Validate Bronze before advancing the watermark

The control watermark must never advance merely because extraction succeeded.

We first verify:

```text
expected extracted rows
        =
rows stored for batch
```

Only then do we commit the new control state.


In [0]:
if extracted_count > 0 and not batch_already_complete:
    bronze_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {BRONZE_TABLE}
        WHERE _batch_id = '{batch_id}'
    """).collect()[0]["cnt"]

    print("Expected rows:", extracted_count)
    print("Bronze rows  :", bronze_count)

    if bronze_count != extracted_count:
        raise RuntimeError(
            f"Bronze validation failed for {batch_id}: "
            f"expected {extracted_count}, found {bronze_count}."
        )

    print("Bronze validation passed.")


## 12 — Commit the latest successful control state

This is the state transition:

```text
EXTRACTED
   ↓
BRONZE VALIDATED
   ↓
CONTROL = SUCCESS
```

`MERGE` is used because `ingestion_state` is a current-state table: one row per source table. Databricks Delta supports `MERGE` for updating matched rows and inserting unmatched rows. citeturn0search0turn0search1


In [0]:
if extracted_count > 0 and not batch_already_complete:
    control_update = spark.createDataFrame(
        [(
            SOURCE_SYSTEM,
            SOURCE_TABLE,
            end_watermark,
            batch_id,
            run_id,
            "SUCCESS",
            extracted_count
        )],
        """
        source_system STRING,
        source_table STRING,
        last_watermark TIMESTAMP,
        last_batch_id STRING,
        last_run_id STRING,
        last_status STRING,
        last_row_count BIGINT
        """
    ).withColumn(
        "last_processed_at",
        F.current_timestamp()
    )

    control_update.createOrReplaceTempView("control_update")

    spark.sql(f"""
    MERGE INTO {CONTROL_TABLE} AS target
    USING control_update AS source
    ON target.source_system = source.source_system
       AND target.source_table = source.source_table

    WHEN MATCHED THEN UPDATE SET
        target.last_watermark = source.last_watermark,
        target.last_batch_id = source.last_batch_id,
        target.last_run_id = source.last_run_id,
        target.last_status = source.last_status,
        target.last_processed_at = source.last_processed_at,
        target.last_row_count = source.last_row_count

    WHEN NOT MATCHED THEN INSERT (
        source_system,
        source_table,
        last_watermark,
        last_batch_id,
        last_run_id,
        last_status,
        last_processed_at,
        last_row_count
    )
    VALUES (
        source.source_system,
        source.source_table,
        source.last_watermark,
        source.last_batch_id,
        source.last_run_id,
        source.last_status,
        source.last_processed_at,
        source.last_row_count
    )
    """)

    print("Control state committed.")


## 13 — Write the ingestion audit record

The log records the execution attempt. A production orchestration layer should also write `FAILED` records when exceptions are caught.

For a recovery scenario, the desired history is:

```text
same batch_id | Run 1 | FAILED
same batch_id | Run 2 | SUCCESS
```

Bronze should still contain the batch only once.


In [0]:
if extracted_count > 0 and not batch_already_complete:
    log_row = spark.createDataFrame(
        [(
            SOURCE_SYSTEM,
            SOURCE_TABLE,
            batch_id,
            run_id,
            start_watermark,
            end_watermark,
            "SUCCESS",
            extracted_count
        )],
        """
        source_system STRING,
        source_table STRING,
        batch_id STRING,
        run_id STRING,
        start_watermark TIMESTAMP,
        end_watermark TIMESTAMP,
        status STRING,
        row_count BIGINT
        """
    ).withColumn(
        "batch_started_at",
        F.current_timestamp()
    ).withColumn(
        "batch_completed_at",
        F.current_timestamp()
    )

    log_row.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

    print("SUCCESS audit record written.")


## 14 — Validation / Monitoring

In [0]:
print("========== PIPELINE VALIDATION ==========")

print("\nLatest control state:")
spark.sql(f"""
SELECT *
FROM {CONTROL_TABLE}
WHERE source_system = '{SOURCE_SYSTEM}'
  AND source_table = '{SOURCE_TABLE}'
""").show(truncate=False)

print("\nRecent ingestion attempts:")
spark.sql(f"""
SELECT *
FROM {LOG_TABLE}
WHERE source_system = '{SOURCE_SYSTEM}'
  AND source_table = '{SOURCE_TABLE}'
ORDER BY batch_started_at DESC
LIMIT 10
""").show(truncate=False)

print("\nBronze batch counts:")
spark.sql(f"""
SELECT
    _batch_id,
    _run_id,
    COUNT(*) AS row_count,
    MIN(_ingestion_timestamp) AS first_ingestion,
    MAX(_ingestion_timestamp) AS last_ingestion
FROM {BRONZE_TABLE}
GROUP BY _batch_id, _run_id
ORDER BY last_ingestion DESC
""").show(truncate=False)

print("==========================================")


# Failure Recovery Playbook

If Bronze succeeds but the control commit fails:

```text
PostgreSQL
    ↓
Batch window determined
    ↓
Bronze append succeeds
    ↓
Control commit fails
    ↓
Pipeline stops
```

On retry:

1. Read the same control watermark.
2. Recalculate the same bounded source window.
3. Reconstruct the same deterministic `batch_id`.
4. Check Bronze for that batch.
5. If it already exists, **do not append it again**.
6. Validate the existing Bronze row count.
7. Commit the control watermark.
8. Record the retry as a successful run.

This avoids duplicate Bronze ingestion without pretending that Bronze and control updates are one atomic cross-table transaction.


# Limitations and next stage

### This is incremental ingestion, not CDC

`updated_at` can detect inserts/updates when the source maintains it correctly. A hard delete is invisible to:

```sql
WHERE updated_at > watermark
```

True CDC requires source-side change capture, including deletes.

### Timestamp-only watermark

If multiple source rows can share an identical timestamp at the precision available, use a composite watermark such as:

```text
(updated_at, primary_key)
```

or a source-native monotonically increasing change identifier.

### Source consistency

The upper watermark creates a deterministic processing window, but it is not a full source database snapshot transaction.

### Security

Move PostgreSQL credentials to Databricks secret/credential management before presenting this as production deployment.

### Next layer

After this ingestion pipeline is stable:

```text
Bronze
  ↓
Silver
  ↓
Current state + SCD Type 2
  ↓
Gold
```

True CDC can then be introduced separately if the project requires source-side INSERT/UPDATE/DELETE events.
